<a href="https://colab.research.google.com/github/dineshaimldev/Computer-Vision/blob/main/Numberplate_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

In [ ]:


from roboflow import Roboflow
rf = Roboflow(api_key="CE1IXnTEurSgHU1fFHXD")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(11)
dataset = version.download("yolov8")


In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO
import shutil, os

In [ ]:
model = YOLO("yolov8n.pt")
result = model.train(
    data = dataset.location + "/data.yaml",
    epochs = 5,
    imgsz = 640,
    batch = 16,
    workers = 2,
    device = 0

)
os.makedirs("saved_models",exist_ok=True)
shutil.copy("runs/detect/train/weights/best.pt","saved_models/license_plate_last.pt")
shutil.copy("runs/detect/train/weights/last.pt","saved_models/license_plate_last.pt")
print("weights saved in saved_models/")

In [ ]:
!pip install easyocr

In [ ]:
!

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
import re
from collections import defaultdict,deque

In [ ]:
model = YOLO("saved_models/license_plate_last.pt")
reader = easyocr.Reader(['en'],gpu = True)
plate_pattern = re.compile("^[A-Z]{2}[0-9]{2}[A-Z]{0,1}[0-9]{4}$")

In [ ]:
def correct_plate_format(ocr_text):
  mapping_num_to_alpha = {"0":"O","1":"I","5":"S","8":"B"}
  mapping_alpha_to_num = {"O":"0","I":"1","Z":"2","S":"5","B":"8"}
  ocr_text = ocr_text.upper().replace(" ", "")
  if len(ocr_text) !=7:
    return ""
  corrected = []
  for i,ch in enumerate(ocr_text):
    if i < 2 or i >=4:
      if ch.isdigit():
        corrected.append(mapping_num_to_alpha[ch])
      elif ch.isaplha():
        corrected.append(ch)
      else:
        return ""
    else:
      if ch.isaplha() and ch in mapping_alpha_to_num:
        corrected.append(mapping_alpha_to_num[ch])
      elif ch.isdigit():
        corrected.append(ch)
      else:
        return ""
  return "".join(corrected)


In [ ]:
def recognize_plate(plate_crop):
  if plate_crop.size == 0:
    return ""
  plate_crop_gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
  _, plate_thresh = cv2.threshold(plate_crop_gray, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  plate_resized = cv2.resize(plate_thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
  try:
    ocr_result = reader.readtext(
        plate_resized, detail=0, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    )
    if len(ocr_result) > 0:
      if plate_pattern.match(ocr_result[0]):
        return ocr_result[0]
    return ""
  except Exception as e:
    print(f"Error during OCR: {e}")
    return ""

In [ ]:
plate_history = defaultdict(lambda:deque(maxlen=10))
plate_final = {}
def get_box_id(x1,y1,x2,y2):
  return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"
def get_stable_plate(box_id,new_text):
  if new_text:
    plate_history[box_id].append(new_text)
    most_common = max(set(plate_history[box_id]),key=plate_history[box_id].count)
    plate_final[box_id] = most_common
  return plate_final.get(box_id,"")

In [ ]:
from prompt_toolkit import output
input_video = "vechicle_video.mp4"
output_video = "output.mp4"
cap = cv2.VideoCapture(input_video)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))
CONF_THRESH = 0.3

In [ ]:
# Download a highly compatible native MP4 traffic video stream
!wget -O vechicle_video.mp4 https://vjs.zencdn.net/v/oceans.mp4

In [ ]:
# Re-initialize capture and writer
import cv2

input_video = "vechicle_video.mp4"
output_video = "output.mp4"

cap = cv2.VideoCapture(input_video)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) if cap.get(cv2.CAP_PROP_FPS) > 0 else 20.0

# Fall back to standard mp4v codec for standard mp4 video generation
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
CONF_THRESH = 0.3
print(f"Video initialized: {width}x{height} @ {fps} FPS")

In [ ]:
while cap.isOpened():
  ret,frame = cap.read()
  if not ret:
    break
  results = model(frame, verbose=False)
  for r in results:
    boxes = r.boxes
    for box in boxes:
      conf = float(box.conf.cpu().numpy())
      if conf > CONF_THRESH:
        x1,y1,x2,y2 = map(int,box.xyxy.cpu().numpy()[0])
        plate_crop = frame[y1:y2,x1:x2]
        text = recognize_plate(plate_crop)
        box_id = get_box_id(x1,y1,x2,y2)
        text = get_stable_plate(box_id,text)
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),3)
        if plate_crop.size > 0:
          overlay_h,overlay_w = 150,400
          plate_resized = cv2.resize(plate_crop,(overlay_w,overlay_h))
          oy1 = max(0,y1 - overlay_h - 40)
          ox1 = x1
          oy2,ox2 = oy1 + overlay_h,ox1 + overlay_w
          if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
            frame[oy1:oy2,ox1:ox2] = plate_resized
            if text:
              cv2.putText(frame,text,(ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(0,0,0),6)
              cv2.putText(frame,text,(ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(255,255,255),3)
  out.write(frame)

In [ ]:
cap.release()
out.release()

print("Annotated video saved as",output_video)

In [ ]:
from IPython.display import HTML
import base64

with open(output_video, 'rb') as f:
    video_bytes = f.read()
    video_base64 = base64.b64encode(video_bytes).decode('utf-8')

HTML(f'''
    <video width="80%" controls>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
''')

In [ ]:
while cap.isOpened():
  ret,frame = cap.read()
  if not ret:
    break
  results = model(frame, verbose=False)
  for r in results:
    boxes = r.boxes
    for box in boxes:
      conf = float(box.conf.cpu().numpy())
      if conf > CONF_THRESH:
        x1,y1,x2,y2 = map(int,box.xyxy.cpu().numpy()[0])
        plate_crop = frame[y1:y2,x1:x2]
        text = recognize_plate(plate_crop)
        box_id = get_box_id(x1,y1,x2,y2)
        text = get_stable_plate(box_id,text)
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),3)
        if plate_crop.size > 0:
          overlay_h,overlay_w = 150,400
          plate_resized = cv2.resize(plate_crop,(overlay_w,overlay_h))
          oy1 = max(0,y1 - overlay_h - 40)
          ox1 = x1
          oy2,ox2 = oy1 + overlay_h,ox1 + overlay_w
          if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
            frame[oy1:oy2,ox1:ox2] = plate_resized
            if text:
              cv2.putText(frame,text,(ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(0,0,0),6)
              cv2.putText(frame,text,(ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(255,255,255),3)
  out.write(frame)

In [ ]:
cap.release()
out.release()

print("Annotated video saved as",output_video)